# 3 · Grupos de alumnos (aprendizaje no supervisado)

**Pregunta de negocio:** ¿hay "tipos" de alumnos según cómo usan el aula y cómo les va, que le sirvan al docente
para decidir a quién acompañar y cómo?

No hay una etiqueta para aprender: es un problema de **clustering**. El notebook recorre:

1. Variables y su preparación (asimetría, escalas por materia, correlación).
2. Estructura de los datos con PCA.
3. Elección de k con cuatro criterios: codo (inercia), silhouette, Davies-Bouldin y Calinski-Harabasz.
4. Los grupos de K-Means: perfiles, centroides y mapa.
5. Comparación con otros algoritmos (jerárquico, mezcla gaussiana, DBSCAN).
6. Estabilidad (bootstrap y semillas).
7. Validación externa: ¿los grupos dicen algo de la condición y del abandono?
8. Alumnos atípicos.

El código de la app está en **`src/ml_grupos.py`** (D57, D68, D71); acá se lo llama y se lo analiza.

In [1]:
import os, warnings
from pathlib import Path
if not Path("src").exists():          # el notebook está en notebooks/: subo a la raíz
    os.chdir("..")
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score,
                             silhouette_samples, silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

from src import metricas as m, ml_abandono as ab, ml_grupos as g
from src.config import FECHA_CORTE_DEFECTO, GRUPOS_LOG, GRUPOS_SEMILLA, GRUPOS_VARIABLES

pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
d = m.cargar()
t = m.tabla_alumnos(d, FECHA_CORTE_DEFECTO)
t = t[g._con_evaluaciones(t)].reset_index(drop=True)
z = g.matriz(t)
print(f"Fecha de corte {pd.Timestamp(FECHA_CORTE_DEFECTO).date()}: {len(t)} alumno-materia, variables {GRUPOS_VARIABLES}")

Fecha de corte 2025-10-24: 637 alumno-materia, variables ['sesiones', 'minutos', 'semanas_activas', 'avance', 'nota_pct']


## 1. Variables y preparación

Cinco variables de `tabla_alumnos`: **sesiones, minutos, semanas con actividad, avance** (reporte de participación)
y **nota** a la fecha (lo no rendido cuenta 0). Cada fila es un alumno en una materia.

In [2]:
desc = t[GRUPOS_VARIABLES].astype(float).describe().T[["mean", "std", "min", "50%", "max"]]
desc["asimetria"] = t[GRUPOS_VARIABLES].astype(float).skew()
desc.round(2)

,mean,std,min,50%,max,asimetria
sesiones,31.01,23.50,0.0,26.00,175.0,1.72
minutos,220.42,187.74,0.0,181.40,1545.7,1.94
semanas_activas,8.39,3.29,0.0,9.00,12.0,-0.83
avance,62.74,28.86,0.0,70.00,100.0,-0.65
nota_pct,48.86,33.66,0.0,55.56,100.0,-0.15


In [3]:
largo = t.melt(id_vars="asignatura", value_vars=GRUPOS_VARIABLES)
px.box(largo, x="asignatura", y="value", facet_col="variable", facet_col_wrap=5, points=False, height=380,
       title="Distribución por materia (unidades reales)").update_yaxes(matches=None, showticklabels=True).show()

Dos problemas para K-Means, que usa distancias:

* **Escalas y asimetría:** sesiones y minutos tienen colas largas (pocos alumnos con muchísimo uso). Sin
  transformar, dominarían la distancia. Se les aplica `log(1 + x)`.
* **Diferencias entre materias:** 150 minutos no significan lo mismo en PyE que en Álgebra, y en AED las notas son
  más bajas. Se estandariza **dentro de cada materia**: cada alumno se compara con su curso.

In [4]:
x = t[GRUPOS_VARIABLES].astype(float).copy()
for c in GRUPOS_LOG:
    x[c] = np.log1p(x[c])
pd.DataFrame({"asimetría original": t[GRUPOS_VARIABLES].astype(float).skew(),
              "con log": x.skew(), "estandarizada por materia": z.skew()}).round(2)

,asimetría original,con log,estandarizada por materia
sesiones,1.72,-1.05,-1.10
minutos,1.94,-1.74,-1.83
semanas_activas,-0.83,-0.83,-0.83
avance,-0.65,-0.65,-0.69
nota_pct,-0.15,-0.15,-0.18


In [5]:
corr = z.corr(method="spearman")
px.imshow(corr.round(2), text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, height=420,
          title="Correlación entre variables (ya transformadas)").show()

Las cuatro variables de actividad están muy correlacionadas entre sí y bastante con la nota: en la práctica hay
pocas dimensiones "reales". Eso se ve mejor con PCA.

## 2. Estructura con PCA

In [6]:
pca = PCA().fit(z)
var = pd.DataFrame({"componente": [f"CP{i + 1}" for i in range(len(GRUPOS_VARIABLES))],
                    "varianza_pct": (100 * pca.explained_variance_ratio_).round(1)})
var["acumulada_pct"] = var["varianza_pct"].cumsum().round(1)
display(var)
pd.DataFrame(pca.components_[:3].T, index=GRUPOS_VARIABLES, columns=["CP1", "CP2", "CP3"]).round(2)

,componente,varianza_pct,acumulada_pct
0,CP1,79.0,79.0
1,CP2,10.6,89.6
2,CP3,5.0,94.6
3,CP4,4.1,98.7
4,CP5,1.3,100.0


,CP1,CP2,CP3
sesiones,0.47,-0.37,-0.19
minutos,0.45,-0.41,0.39
semanas_activas,0.46,-0.16,-0.61
avance,0.45,0.27,0.63
nota_pct,0.40,0.77,-0.23


El primer componente (≈ "cuánto participa y le va", todas las cargas del mismo signo) explica la mayor parte de la
variación; el segundo separa **nota** de **actividad** (alumnos que entran poco con buena nota, o al revés). Con dos
componentes se ve casi todo: por eso la app dibuja el mapa de grupos en esos dos ejes.

## 3. ¿Cuántos grupos?

Cuatro criterios para k = 2 a 8. No hay un "k verdadero": se busca uno que tenga grupos separados **y** útiles.

In [7]:
filas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=GRUPOS_SEMILLA).fit(z)
    filas.append({"k": k, "inercia": km.inertia_, "silhouette": silhouette_score(z, km.labels_),
                  "davies_bouldin": davies_bouldin_score(z, km.labels_),
                  "calinski_harabasz": calinski_harabasz_score(z, km.labels_),
                  "grupo_mas_chico": np.bincount(km.labels_).min()})
ks = pd.DataFrame(filas).round(3)
display(ks)
fig = px.line(ks.melt(id_vars="k", value_vars=["inercia", "silhouette", "davies_bouldin", "calinski_harabasz"]),
              x="k", y="value", facet_col="variable", facet_col_wrap=4, markers=True, height=300,
              title="Codo (inercia ↓), silhouette (↑), Davies-Bouldin (↓) y Calinski-Harabasz (↑)")
fig.update_yaxes(matches=None, showticklabels=True).show()

,k,inercia,silhouette,davies_bouldin,calinski_harabasz,grupo_mas_chico
0,2,1470.447,0.475,0.833,740.415,208
1,3,982.497,0.406,0.918,710.632,84
2,4,820.406,0.332,1.122,608.150,48
3,5,716.155,0.299,1.164,544.683,44
4,6,626.574,0.294,1.090,515.299,23
5,7,575.386,0.245,1.190,476.218,23
6,8,534.099,0.227,1.238,445.989,22


In [8]:
g.evaluar_k(d, FECHA_CORTE_DEFECTO)     # lo mismo que usa la app (k = 2 de referencia, candidatos 3 a 6)

,k,silhouette,davies_bouldin,calinski_harabasz,inercia,candidato,tamanos,elegido
0,2,0.475,0.833,740.4,1470.4,False,"[429, 208]",False
1,3,0.406,0.918,710.6,982.5,True,"[342, 211, 84]",True
2,4,0.332,1.122,608.1,820.4,True,"[296, 178, 115, 48]",False
3,5,0.299,1.164,544.7,716.2,True,"[250, 117, 116, 110, 44]",False
4,6,0.294,1.090,515.3,626.6,True,"[213, 128, 107, 93, 73, 23]",False


* k = 2 tiene el mejor silhouette, pero solo separa "entran / no entran", que ya lo dice el estado de actividad
  (D35): no aporta. Por eso la app elige entre 3 y 6 (D57).
* Entre 3 y 6, **los tres índices coinciden en k = 3** (mayor silhouette, menor Davies-Bouldin, mayor
  Calinski-Harabasz). El codo no es nítido: la inercia baja de forma gradual, típico de datos sin grupos muy marcados.
* Un silhouette de ~0,4 indica una estructura **moderada**: hay grupos, pero con alumnos en las fronteras.

### Silhouette por alumno

In [9]:
km3 = KMeans(n_clusters=3, n_init=20, random_state=GRUPOS_SEMILLA).fit(z)
sil = pd.DataFrame({"grupo": km3.labels_, "silhouette": silhouette_samples(z, km3.labels_)})
sil = sil.sort_values(["grupo", "silhouette"]).reset_index(drop=True)
px.bar(sil, x=sil.index, y="silhouette", color=sil["grupo"].astype(str), height=320,
       title="Silhouette de cada alumno (negativo = estaría mejor en otro grupo)").update_xaxes(visible=False).show()
print(f"Alumnos con silhouette negativo: {(sil.silhouette < 0).mean():.1%}")

Alumnos con silhouette negativo: 1.1%


## 4. Los grupos de K-Means (lo que muestra la app)

Los nombres salen del **centroide** de cada grupo (cuánta actividad y qué nota tiene respecto del promedio de su
materia), no del número arbitrario de K-Means (D57, D68).

In [10]:
perfil = g.perfil_grupos(d, FECHA_CORTE_DEFECTO)
perfil.drop(columns="descripcion_grupo")

,grupo,nombre_grupo,alumnos,pct,mediana_sesiones,mediana_minutos,mediana_semanas_activas,mediana_avance,mediana_nota_pct,nota_promedio,pct_activos,pct_libre,pct_promociona,pct_regular
0,1,Comprometidos,342,53.7,39.0,270.1,11.0,86.0,76.0,73.5,78.9,17.5,44.2,38.3
1,2,Intermitentes,211,33.1,18.0,134.7,7.0,47.0,20.0,25.3,27.5,87.2,6.6,6.2
2,3,Desconectados,84,13.2,4.0,15.4,2.0,9.0,0.0,7.8,4.8,98.8,0.0,1.2


In [11]:
cen = g.centroides(d, FECHA_CORTE_DEFECTO)
px.imshow(cen.pivot(index="nombre_grupo", columns="descripcion", values="desvios"), text_auto=".2f",
          color_continuous_scale="RdBu", zmin=-2, zmax=2, height=320,
          title="Centroides en desvíos respecto del promedio de la materia").show()
mapa = g.mapa_grupos(d, FECHA_CORTE_DEFECTO)
px.scatter(mapa["alumnos"], x="eje1", y="eje2", color="nombre_grupo", symbol="asignatura", opacity=0.7, height=480,
           labels={"eje1": f"Eje 1: actividad y nota ({mapa['varianza'][0]} %)",
                   "eje2": f"Eje 2: nota vs. actividad ({mapa['varianza'][1]} %)"},
           title="Grupos en los dos primeros componentes").show()

In [12]:
g.perfil_grupos(d, FECHA_CORTE_DEFECTO, por_materia=True)[["asignatura", "nombre_grupo", "alumnos", "pct",
                                                            "nota_promedio", "pct_activos"]]

,asignatura,nombre_grupo,alumnos,pct,nota_promedio,pct_activos
0,Algoritmos y Estructuras de Datos,Comprometidos,96,47.1,66.4,82.3
1,Algoritmos y Estructuras de Datos,Intermitentes,82,40.2,15.7,24.4
2,Algoritmos y Estructuras de Datos,Desconectados,26,12.7,5.5,0.0
3,Probabilidad y Estadística,Comprometidos,117,54.2,69.7,82.9
4,Probabilidad y Estadística,Intermitentes,70,32.4,22.2,30.0
5,Probabilidad y Estadística,Desconectados,29,13.4,7.8,6.9
6,Álgebra,Comprometidos,129,59.4,82.2,72.9
7,Álgebra,Intermitentes,59,27.2,42.3,28.8
8,Álgebra,Desconectados,29,13.4,10.0,6.9


### ¿Y con más grupos?

Si el docente fuerza k = 5, aparecen subgrupos con nombre propio (D68): por ejemplo "Muy comprometidos" y
"Comprometidos". Sirve para afinar, pero los grupos chicos son menos estables (sección 6).

In [13]:
g.perfil_grupos(d, FECHA_CORTE_DEFECTO, k=5)[["grupo", "nombre_grupo", "alumnos", "pct", "nota_promedio",
                                              "pct_activos"]]

,grupo,nombre_grupo,alumnos,pct,nota_promedio,pct_activos
0,1,Muy comprometidos,250,39.2,76.0,86.8
1,2,Comprometidos,117,18.4,68.9,44.4
2,3,Esforzados con nota baja,116,18.2,16.0,44.8
3,4,Intermitentes,110,17.3,19.1,9.1
4,5,Desconectados,44,6.9,2.4,2.3


## 5. Comparación con otros algoritmos

Mismos datos y k = 3. El **índice de Rand ajustado (ARI)** mide cuánto coinciden dos agrupamientos (1 = iguales,
0 = como al azar).

* **Jerárquico (Ward):** une grupos de abajo hacia arriba minimizando la varianza; no depende de la semilla.
* **Mezcla gaussiana (GMM):** grupos elípticos y asignación probabilística.
* **DBSCAN:** grupos por densidad; no pide k y marca "ruido". El `eps` se elige con el gráfico de distancia al
  k-ésimo vecino.

In [14]:
vec = np.sort(NearestNeighbors(n_neighbors=10).fit(z).kneighbors(z)[0][:, -1])
px.line(y=vec, labels={"index": "alumnos ordenados", "y": "distancia al 10.º vecino"}, height=280,
        title="Elegir eps de DBSCAN: el 'codo' de esta curva").show()

In [15]:
algoritmos = {
    "K-Means": km3.labels_,
    "Jerárquico (Ward)": AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(z),
    "Mezcla gaussiana": GaussianMixture(n_components=3, n_init=5, random_state=GRUPOS_SEMILLA).fit(z).predict(z),
    "DBSCAN (eps 0,9)": DBSCAN(eps=0.9, min_samples=10).fit_predict(z),
}
filas = []
for nombre, et in algoritmos.items():
    sin_ruido = et >= 0
    n = len(set(et[sin_ruido]))
    filas.append({"algoritmo": nombre, "grupos": n, "ruido_pct": round(100 * (~sin_ruido).mean(), 1),
                  "silhouette": round(silhouette_score(z[sin_ruido], et[sin_ruido]), 3) if n > 1 else np.nan,
                  "ARI vs K-Means": round(adjusted_rand_score(km3.labels_, et), 3),
                  "tamaños": sorted(np.bincount(et[sin_ruido]).tolist(), reverse=True)})
pd.DataFrame(filas)

,algoritmo,grupos,ruido_pct,silhouette,ARI vs K-Means,tamaños
0,K-Means,3,0.0,0.406,1.000,"[342, 211, 84]"
1,Jerárquico (Ward),3,0.0,0.375,0.694,"[383, 169, 85]"
2,Mezcla gaussiana,3,0.0,0.294,0.505,"[273, 260, 104]"
3,"DBSCAN (eps 0,9)",1,5.2,NaN,0.053,[604]


El jerárquico coincide bastante con K-Means (ARI ~0,7) y la mezcla gaussiana menos (~0,5): el "tramo alto / medio /
bajo" aparece con todos, pero **las fronteras entre grupos cambian según el algoritmo**, lo esperable si los alumnos
forman un continuo. K-Means tiene el mejor silhouette de los tres. DBSCAN, en cambio, encuentra sobre todo **un gran bloque denso** y algo de ruido: no hay
grupos separados por zonas vacías, sino un continuo de "más a menos participación". K-Means sirve para **segmentar**
ese continuo en tramos interpretables, que es lo que se busca.

## 6. Estabilidad

Si los grupos fueran un accidente de la muestra, cambiarían mucho al remuestrear. `g.estabilidad()` (D71) remuestrea a
los alumnos con reposición 30 veces, reajusta y compara con el ARI.

In [16]:
est = pd.concat([g.estabilidad(d, FECHA_CORTE_DEFECTO, k=k, repeticiones=30) for k in [3, 4, 5, 6]])
display(est.groupby("k")["ari"].describe()[["mean", "std", "min", "50%"]].round(3))
px.box(est, x="k", y="ari", points="all", height=330, title="Estabilidad bootstrap (ARI con el modelo original)").show()

,mean,std,min,50%
k,,,,
3,0.949,0.058,0.795,0.976
4,0.885,0.085,0.728,0.921
5,0.762,0.140,0.543,0.776
6,0.872,0.088,0.597,0.903


In [17]:
semillas = [KMeans(n_clusters=3, n_init=1, random_state=s).fit(z).labels_ for s in range(20)]
ari_sem = [adjusted_rand_score(km3.labels_, s) for s in semillas]
print(f"Con una sola inicialización, ARI contra el modelo de la app: media {np.mean(ari_sem):.3f}, mín. {min(ari_sem):.3f}")

Con una sola inicialización, ARI contra el modelo de la app: media 1.000, mín. 1.000


Con k = 3 los grupos son muy estables (ARI ~0,95): con otros alumnos de la misma cursada se obtendrían los mismos
grupos. Con más grupos la estabilidad baja (k = 5 es el más inestable): los subgrupos de k grande dependen más de la
muestra. Con k = 3 ni siquiera importa la inicialización (todas las semillas dan lo mismo); igual la app usa
`n_init=20` por si cambian los datos.

## 7. Validación externa: ¿los grupos dicen algo útil?

Los grupos se armaron sin mirar la condición ni el abandono. Si son útiles, deberían diferenciarse en ambas.

In [18]:
gr = g.grupos_alumnos(d, FECHA_CORTE_DEFECTO)
pd.crosstab(gr["nombre_grupo"], gr["condicion"], normalize="index").mul(100).round(1)

condicion,Libre,Promociona,Regular
nombre_grupo,,,
Comprometidos,17.5,44.2,38.3
Desconectados,98.8,0.0,1.2
Intermitentes,87.2,6.6,6.2


Para el abandono se arman los grupos a una fecha con desenlace conocido (20/09: hay más de 28 días de datos después)
y se mira qué proporción de cada grupo abandonó la materia (D67).

In [19]:
corte = pd.Timestamp("2025-09-20")
g_corte = g.grupos_alumnos(d, corte)
desenlace = ab.dataset(d, cortes=[corte])[ab.CLAVES + ["abandona"]]
x = g_corte.merge(desenlace, on=ab.CLAVES, how="left")
x["abandona"] = x["abandona"].astype("boolean")
res = x.groupby("nombre_grupo").agg(alumnos=("id_alumno", "size"), entraron=("abandona", "count"),
                                    abandonaron_pct=("abandona", "mean"))
res["abandonaron_pct"] = (100 * res["abandonaron_pct"]).round(1)
res.sort_values("abandonaron_pct")

,alumnos,entraron,abandonaron_pct
nombre_grupo,,,
Comprometidos,343,343,0.9
Intermitentes,215,215,21.9
Desconectados,79,70,52.9


## 8. Alumnos atípicos

`IsolationForest` marca al 3 % de alumnos con combinaciones menos comunes. No son errores de datos (eso se revisó en
el notebook 1).

In [20]:
iso = IsolationForest(contamination=0.03, random_state=GRUPOS_SEMILLA).fit(z)
t["atipico"] = iso.predict(z) == -1
t["grupo"] = gr.set_index(ab.CLAVES).loc[pd.MultiIndex.from_frame(t[ab.CLAVES]), "nombre_grupo"].to_numpy()
print(f"{t.atipico.sum()} atípicos de {len(t)}")
display(pd.crosstab(t["grupo"], t["atipico"]))
t.loc[t.atipico, ["asignatura", "id_alumno", "grupo", *GRUPOS_VARIABLES, "condicion"]].sort_values("grupo")

20 atípicos de 637


atipico,False,True
grupo,,
Comprometidos,342,0
Desconectados,64,20
Intermitentes,211,0


,asignatura,id_alumno,grupo,sesiones,minutos,semanas_activas,avance,nota_pct,condicion
19,Algoritmos y Estructuras de Datos,ALU_1276,Desconectados,1,9.0,1,38,20.0,Libre
73,Algoritmos y Estructuras de Datos,ALU_1282,Desconectados,1,0.2,1,1,0.0,Libre
126,Algoritmos y Estructuras de Datos,ALU_1241,Desconectados,0,0.0,0,0,0.0,Libre
137,Algoritmos y Estructuras de Datos,ALU_1281,Desconectados,1,0.1,1,0,0.0,Libre
186,Algoritmos y Estructuras de Datos,ALU_1163,Desconectados,0,0.0,0,0,0.0,Libre
217,Probabilidad y Estadística,ALU_1210,Desconectados,0,0.0,0,0,0.0,Libre
245,Probabilidad y Estadística,ALU_1016,Desconectados,1,2.3,1,0,0.0,Libre
283,Probabilidad y Estadística,ALU_1282,Desconectados,0,0.0,0,0,0.0,Libre
351,Probabilidad y Estadística,ALU_1281,Desconectados,1,0.6,1,0,0.0,Libre
401,Probabilidad y Estadística,ALU_1163,Desconectados,0,0.0,0,0,0.0,Libre


Los atípicos son casi todos el extremo inferior: alumnos que entraron una vez o nunca y no rindieron. Confirma que el
grupo "Desconectados" es una cola larga y no un grupo compacto; no aparecen perfiles raros dentro de los otros grupos
que justifiquen tratarlos aparte. Por eso no se agregó una alerta de "atípico" a la app.

## Conclusiones

| Hallazgo | Qué se decidió |
|---|---|
| Sesiones y minutos son muy asimétricos (asimetría 1,7 y 1,9) y las escalas cambian entre materias. | `log(1 + x)` en sesiones y minutos y estandarización **dentro de cada materia** (D57): cada alumno se compara con su curso. |
| Las 5 variables están muy correlacionadas: el primer componente de PCA explica el 79 % y el segundo (nota vs. actividad) otro 11 %. | El mapa de la app usa esos dos ejes. Los grupos son, sobre todo, tramos de "cuánto participa y le va". |
| k = 2 tiene el mejor silhouette pero solo separa "entran / no entran". Entre 3 y 6, silhouette, Davies-Bouldin y Calinski-Harabasz coinciden en **k = 3**. El codo no es nítido. | Se mantiene k automático entre 3 y 6 por silhouette (D57). `evaluar_k` informa también Davies-Bouldin y Calinski-Harabasz (D71). |
| Silhouette 0,41: estructura moderada; solo el 1 % de los alumnos tiene silhouette negativo. | Se explica en la app como "grupos con fronteras difusas", no como tipos cerrados. |
| Grupos con k = 3: Comprometidos (54 %), Intermitentes (33 %) y Desconectados (13 %). | Nombres por centroide (D57, D68). |
| Jerárquico (Ward) coincide bastante con K-Means (ARI 0,69), la mezcla gaussiana menos (0,51). DBSCAN encuentra un solo bloque denso más 5 % de ruido. | No hay grupos separados por zonas vacías: K-Means segmenta un continuo. Se queda K-Means (mejor silhouette, simple y estable). |
| Bootstrap: ARI medio 0,95 con k = 3; baja con k = 4 a 6 (0,76 a 0,89). | Nueva función `estabilidad()` en `src/ml_grupos.py` (D71). Los grupos de k = 3 son confiables; los de k grande, menos. |
| Validación externa: abandonó la materia el 1 % de los Comprometidos, el 22 % de los Intermitentes y el 53 % de los Desconectados; el 99 % de los Desconectados está libre. | Los grupos, armados sin mirar la condición ni el abandono, sirven para priorizar. Se usan como complemento del predictor. |
| Los atípicos (IsolationForest) son casi todos alumnos que no entraron nunca o una vez. | No se agrega una alerta de atípicos: ya los cubren "Desconectados" y las alertas de inactividad. |